In [5]:
import pandas as pd
import numpy as np
import os

# Pastikan folder output tersedia
os.makedirs('../data/processed', exist_ok=True)

# Daftar 16 kolom kunci untuk pemodelan PD, LGD, EAD, dan Staging
cols_to_use = [
    'id', 'issue_d', 'term', 'loan_amnt', 'funded_amnt', 'out_prncp',
    'int_rate', 'grade', 'sub_grade', 'emp_length', 'home_ownership',
    'annual_inc', 'dti', 'fico_range_low', 'delinq_2yrs', 'loan_status',
    'recoveries', 'collection_recovery_fee'
]

file_path = '../data/raw/accepted_2007_to_2018Q4.csv'

# Load data (low_memory=False untuk menghindari warning tipe data campuran)
print("Memuat dataset...")
df = pd.read_csv(file_path, usecols=cols_to_use, low_memory=False)
print(f"Total baris awal: {len(df):,} | Total kolom: {df.shape[1]}")

Memuat dataset...
Total baris awal: 2,260,701 | Total kolom: 18


In [6]:
# 1. Hapus baris kosong atau baris footer yang tidak punya loan_amnt / issue_d
df = df.dropna(subset=['loan_amnt', 'issue_d', 'loan_status']).copy()

# 2. Konversi tanggal pencairan (issue_d) ke format datetime
df['issue_date'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')
df = df.dropna(subset=['issue_date']).copy()

# 3. Filter data mulai Januari 2015 ke atas (sekitar 1,5 juta baris yang sangat berkualitas)
df = df[df['issue_date'] >= '2015-01-01'].copy()

# 4. Bersihkan kolom 'term' menjadi angka integer (36 atau 60)
df['tenor_months'] = df['term'].str.extract('(\d+)').astype(int)

# 5. Bersihkan kolom 'emp_length' (lama bekerja) menjadi angka tahun
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
    '10+ years': 10
}
df['emp_years'] = df['emp_length'].map(emp_map).fillna(-1) # -1 untuk Missing/Unknown

# 6. Tangani missing value pada fitur numerik utama
df['dti'] = df['dti'].fillna(df['dti'].median())
df['annual_inc'] = df['annual_inc'].fillna(df['annual_inc'].median())
df['delinq_2yrs'] = df['delinq_2yrs'].fillna(0)
df['recoveries'] = df['recoveries'].fillna(0)
df['collection_recovery_fee'] = df['collection_recovery_fee'].fillna(0)

# Hapus outlier ekstrem pada DTI (misal DTI > 100% atau negatif akibat salah input)
df = df[(df['dti'] >= 0) & (df['dti'] <= 100)].copy()

print(f"Total baris setelah cleaning (2015-2018): {len(df):,}")

Total baris setelah cleaning (2015-2018): 1,791,760


In [7]:
# 1. Klasifikasi Staging PSAK 71:
#    0 = Lunas (Fully Paid) -> Tidak masuk perhitungan CKPN aktif
#    1 = Stage 1 (Performing / Lancar) -> 12-Month ECL
#    2 = Stage 2 (Underperforming / DPD 31-120 atau Grade F/G sebagai proksi SICR) -> Lifetime ECL
#    3 = Stage 3 (Non-Performing / Charged Off atau Default) -> Lifetime ECL (PD = 100%)
conditions_stage = [
    df['loan_status'] == 'Fully Paid',
    df['loan_status'].isin(['Charged Off', 'Default']),
    (df['loan_status'] == 'Late (31-120 days)') | (df['grade'].isin(['F', 'G']))
]
choices_stage = [0, 3, 2]
df['psak71_stage'] = np.select(conditions_stage, choices_stage, default=1)

# 2. Target Biner untuk Pemodelan PD (1 = Gagal Bayar, 0 = Lancar/Lunas)
#    Catatan: Pinjaman 'Current' yang masih berjalan belum diketahui hasil akhirnya,
#    sehingga untuk training model PD nanti kita fokus pada pinjaman yang sudah selesai (Fully Paid vs Charged Off/Default)
df['is_default'] = np.where(
    df['loan_status'].isin(['Charged Off', 'Default', 'Late (31-120 days)']), 1, 0
)

# 3. Perhitungan LGD Empiris Historis (Khusus pinjaman yang sudah Charged Off)
#    Rumus: LGD = 1 - ((Recoveries - Biaya Penagihan) / Dana Dicairkan)
net_recovery = df['recoveries'] - df['collection_recovery_fee']
df['empirical_lgd'] = np.where(
    (df['loan_status'] == 'Charged Off') & (df['funded_amnt'] > 0),
    1.0 - (net_recovery / df['funded_amnt']),
    np.nan
)
df['empirical_lgd'] = df['empirical_lgd'].clip(0, 1)

# Tampilkan ringkasan hasil pre-processing
print("=== Distribusi PSAK 71 Stage ===")
print(df['psak71_stage'].value_counts().sort_index())
print("\n=== Rata-rata LGD Empiris per Grade (Pinjaman Charged Off) ===")
print(df.groupby('grade')['empirical_lgd'].mean().round(4))

=== Distribusi PSAK 71 Stage ===
psak71_stage
0    701799
1    866142
2     31863
3    191956
Name: count, dtype: int64

=== Rata-rata LGD Empiris per Grade (Pinjaman Charged Off) ===
grade
A    0.9508
B    0.9456
C    0.9401
D    0.9372
E    0.9309
F    0.9249
G    0.9225
Name: empirical_lgd, dtype: float64


In [8]:
output_path = '../data/processed/staged_loans.parquet'
df.to_parquet(output_path, index=False)
print(f"Berhasil disimpan ke {output_path}!")

Berhasil disimpan ke ../data/processed/staged_loans.parquet!
